In [20]:
import json
import math
import os

import torch
import torch.nn.functional as F

Weights_path = "pythia-70m/step143000"
DTYPE = torch.float32
torch.set_grad_enabled(False)          # inference only, no autograd anywhere

with open(os.path.join(Weights_path, "metadata.json")) as f:
    meta = json.load(f)

Layer   = meta["num_layers"]               # transformer layers
Hidden_Dimension   = meta["hidden_size"]              # width of the residual stream
Heads  = meta["num_heads"]                # attention heads
Head_Dimension  = meta["head_dim"]                 # per-head width = Hidden_Dimension // Heads
MLP_HH_Dimension  = meta["intermediate_size"]        # MLP inner width
Vocab_Size   = meta["vocab_size"]               # vocab, including padding
Layer_Norm_Eps = meta["layer_norm_eps"]
Rotated_Dimension   = meta["position_encoding"]["rotary_ndims"]   # rotated dims per head
Rope_Theta = meta["position_encoding"]["rope_theta"]

print(f"{meta['model']} @ {meta['revision']}  (step {meta['training_step']}, "
      f"{meta['tokens_seen'] / 1e9:.0f}B tokens seen)")

print(f"Number of layers = {Layer:5d}   layers, each with its own parameters (no sharing)")
print(f"Hidden Dimension = {Hidden_Dimension:5d}   hidden_size")
print(f"Heads = {Heads:5d}   heads")  
print(f"Head Dimension = {Head_Dimension}  = {Hidden_Dimension} / {Heads}")
print(f"MLP Inner Dimension = {MLP_HH_Dimension:5d}   = 4 * {Hidden_Dimension}")
print(f"Vocab Size = {Vocab_Size:5d}   vocab (only 50277 real tokens; the tail is padding)")

print(f"RoPE: first {Rotated_Dimension} of {Head_Dimension} dims per head are rotated, theta = {Rope_Theta}")

print(f"use_parallel_residual = {meta['residual_structure']['use_parallel_residual']}")
print(f"activation = {meta['hidden_act']}   Layer Norm eps = {Layer_Norm_Eps}")

EleutherAI/pythia-70m @ step143000  (step 143000, 300B tokens seen)
Number of layers =     6   layers, each with its own parameters (no sharing)
Hidden Dimension =   512   hidden_size
Heads =     8   heads
Head Dimension = 64  = 512 / 8
MLP Inner Dimension =  2048   = 4 * 512
Vocab Size = 50304   vocab (only 50277 real tokens; the tail is padding)
RoPE: first 16 of 64 dims per head are rotated, theta = 10000
use_parallel_residual = True
activation = gelu   Layer Norm eps = 1e-05


In [21]:
fused_dir = os.path.join(Weights_path, "fused")

W = {}
for name in sorted(os.listdir(fused_dir)):
    if name.endswith(".pt"):
        W[name[:-3]] = torch.load(os.path.join(fused_dir, name), map_location="cpu").to(DTYPE)

# Every expected shape is derived from metadata, not written down as a literal.
expected = {
    "embed_in":            (Vocab_Size, Hidden_Dimension),      # row v is the embedding of token v
    "W_unembed":           (Hidden_Dimension, Vocab_Size),      # untied from embed_in
    "final_ln_weight":     (Hidden_Dimension,),
    "final_ln_bias":       (Hidden_Dimension,),
    "attention_ln_weight": (Layer, Hidden_Dimension),      # input_layernorm
    "attention_ln_bias":   (Layer, Hidden_Dimension),
    "W_Q": (Layer, Hidden_Dimension, Hidden_Dimension), "W_K": (Layer, Hidden_Dimension, Hidden_Dimension), "W_V": (Layer, Hidden_Dimension, Hidden_Dimension), "W_O": (Layer, Hidden_Dimension, Hidden_Dimension),
    "b_Q": (Layer, Hidden_Dimension),    "b_K": (Layer, Hidden_Dimension),    "b_V": (Layer, Hidden_Dimension),    "b_O": (Layer, Hidden_Dimension),
    "ffn_ln_weight":       (Layer, Hidden_Dimension),      # post_attention_layernorm
    "ffn_ln_bias":         (Layer, Hidden_Dimension),
    "W_ffn":        (Layer, Hidden_Dimension, MLP_HH_Dimension),         # up projection
    "b_ffn":        (Layer, MLP_HH_Dimension),
    "W_ffn_output": (Layer, MLP_HH_Dimension, Hidden_Dimension),         # down projection
    "b_ffn_output": (Layer, Hidden_Dimension),
}

assert set(W) == set(expected), f"file set differs: {set(W) ^ set(expected)}"
for key, shape in expected.items():
    assert tuple(W[key].shape) == shape, f"{key}: {tuple(W[key].shape)} != {shape}"
    print(f"  {key:22s} {str(list(W[key].shape)):16s}")

print(f"\n{len(W)} tensors, {sum(t.numel() for t in W.values()) / 1e6:.1f}M parameters, "
      f"all shapes agree with metadata")

  embed_in               [50304, 512]    
  W_unembed              [512, 50304]    
  final_ln_weight        [512]           
  final_ln_bias          [512]           
  attention_ln_weight    [6, 512]        
  attention_ln_bias      [6, 512]        
  W_Q                    [6, 512, 512]   
  W_K                    [6, 512, 512]   
  W_V                    [6, 512, 512]   
  W_O                    [6, 512, 512]   
  b_Q                    [6, 512]        
  b_K                    [6, 512]        
  b_V                    [6, 512]        
  b_O                    [6, 512]        
  ffn_ln_weight          [6, 512]        
  ffn_ln_bias            [6, 512]        
  W_ffn                  [6, 512, 2048]  
  b_ffn                  [6, 2048]       
  W_ffn_output           [6, 2048, 512]  
  b_ffn_output           [6, 512]        

20 tensors, 70.4M parameters, all shapes agree with metadata


In [22]:
per_head_dir = os.path.join(Weights_path, "per_head")
PH = {n[:-3]: torch.load(os.path.join(per_head_dir, n), map_location="cpu").to(DTYPE)
      for n in sorted(os.listdir(per_head_dir)) if n.endswith(".pt")}

for n in ("Q", "K", "V"):
    # [Layer, Heads, Hidden_Dimension, Head_Dimension] -> [Layer, Hidden_Dimension, Heads * Head_Dimension]: concatenate the heads along the output axis
    assert torch.equal(PH[f"W_{n}_heads"].permute(0, 2, 1, 3).reshape(Layer, Hidden_Dimension, Hidden_Dimension), W[f"W_{n}"])
    assert torch.equal(PH[f"b_{n}_heads"].reshape(Layer, Hidden_Dimension), W[f"b_{n}"])
    print(f"  W_{n}_heads / b_{n}_heads  ==  column slices of W_{n} / b_{n}")

# W_O is the other way round: the attention output is the head concatenation, so the
# heads split W_O's INPUT axis. [Layer, Heads, Head_Dimension, Hidden_Dimension] -> [Layer, Heads * Head_Dimension, Hidden_Dimension]
assert torch.equal(PH["W_O_heads"].reshape(Layer, Hidden_Dimension, Hidden_Dimension), W["W_O"])
print("  W_O_heads              ==  row slices of W_O")

# This is the indexing the single-head section below relies on - check it here first.
# A head's slice is always Head_Dimension wide, on W_Q's output axis and W_O's input axis.
h_test = 3
assert torch.equal(W["W_Q"][:, :, h_test * Head_Dimension:(h_test + 1) * Head_Dimension], PH["W_Q_heads"][:, h_test])
assert torch.equal(W["W_O"][:, h_test * Head_Dimension:(h_test + 1) * Head_Dimension, :], PH["W_O_heads"][:, h_test])
print(f"\nSo head h has W_Q = W['W_Q'][layer][:, h*{Head_Dimension}:(h+1)*{Head_Dimension}] and "
      f"W_O = W['W_O'][layer][h*{Head_Dimension}:(h+1)*{Head_Dimension}, :]")

  W_Q_heads / b_Q_heads  ==  column slices of W_Q / b_Q
  W_K_heads / b_K_heads  ==  column slices of W_K / b_K
  W_V_heads / b_V_heads  ==  column slices of W_V / b_V
  W_O_heads              ==  row slices of W_O

So head h has W_Q = W['W_Q'][layer][:, h*64:(h+1)*64] and W_O = W['W_O'][layer][h*64:(h+1)*64, :]


In [23]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(meta["model"])

PROMPT = "The capital of France is Paris, and the capital of Germany is"
ids = tok(PROMPT, return_tensors="pt").input_ids[0]
T = ids.shape[0]

print(f"T = {T} tokens")
print(ids.tolist())
print([tok.decode(i) for i in ids])

/Users/max/anaconda3/envs/loopeddynamics/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


T = 13 tokens
[510, 5347, 273, 6181, 310, 7785, 13, 285, 253, 5347, 273, 6176, 310]
['The', ' capital', ' of', ' France', ' is', ' Paris', ',', ' and', ' the', ' capital', ' of', ' Germany', ' is']


In [24]:
h_embed = W["embed_in"][ids]          # [T, Hidden_Dimension]

print(f"h = embed_in[ids] : {list(h_embed.shape)}")
print("norm per position : " + "  ".join(f"{n:.3f}" for n in h_embed.norm(dim=-1)))
print("Note: at this point the same token at different positions has an identical vector -")
print("      positions 1 and 9 are both ' capital' and their norms match exactly.")

h = embed_in[ids] : [13, 512]
norm per position : 0.616  0.726  0.516  0.712  0.556  0.718  0.519  0.481  0.532  0.726  0.516  0.714  0.556
Note: at this point the same token at different positions has an identical vector -
      positions 1 and 9 are both ' capital' and their norms match exactly.


In [25]:
def layer_norm(x, weight, bias, eps=Layer_Norm_Eps):
    """LayerNorm over the last axis. Biased variance, matching PyTorch."""
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    return (x - mu) / torch.sqrt(var + eps) * weight + bias


ours = layer_norm(h_embed, W["attention_ln_weight"][0], W["attention_ln_bias"][0])
ref_ = F.layer_norm(h_embed, (Hidden_Dimension,), W["attention_ln_weight"][0], W["attention_ln_bias"][0], Layer_Norm_Eps)

print(f"max diff vs F.layer_norm : {(ours - ref_).abs().max():.2e}")
assert torch.allclose(ours, ref_, atol=1e-6)
print("LayerNorm ok")

max diff vs F.layer_norm : 9.54e-07
LayerNorm ok


## 7. RoPE (rotary position embedding, first 16 dims of each head only $25\%$)

For position $m$ and frequency $\theta_i = \mathrm{base}^{-2i/d_{rot}}$, dimensions $i$ and
$i + d_{rot}/2$ are paired and rotated in 2D:

$$\begin{pmatrix} x'_i \\ x'_{i+d/2}\end{pmatrix} =
\begin{pmatrix}\cos m\theta_i & -\sin m\theta_i\\ \sin m\theta_i & \cos m\theta_i\end{pmatrix}
\begin{pmatrix} x_i \\ x_{i+d/2}\end{pmatrix}$$

In code that is `x * cos + rotate_half(x) * sin`, where `rotate_half([a, b]) = [-b, a]` splitting into
halves.

**The Pythia trap**: `partial_rotary_factor = 0.25`, so only the first `ROT = 16` dims of each head
are rotated and the other 48 pass through. Rotating all 64 moves the logits by ~2e+01 and flips the
argmax.

In [26]:
def rope_tables(seq_len):
    """Return two [T, Rotated_Dimension] tables. Built in float32 then cast, as transformers does."""
    half = torch.arange(0, Rotated_Dimension, 2, dtype=torch.float32)     # 0, 2, ..., Rotated_Dimension-2
    inv_freq = 1.0 / (Rope_Theta ** (half / Rotated_Dimension))                # [Rotated_Dimension/2]
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)   # [T, Rotated_Dimension/2]
    emb = torch.cat([freqs, freqs], dim=-1)                 # [T, Rotated_Dimension], halves share frequencies
    return emb.cos().to(DTYPE), emb.sin().to(DTYPE)


def apply_rope(x, cos, sin):
    """x: [T, Heads, Head_Dimension]. Rotate the first Rotated_Dimension dims, pass the rest through. cos/sin broadcast over heads."""
    rot, keep = x[..., :Rotated_Dimension], x[..., Rotated_Dimension:]
    half = Rotated_Dimension // 2
    rotated = torch.cat([-rot[..., half:], rot[..., :half]], dim=-1)   # rotate_half
    rot = rot * cos[:, None, :] + rotated * sin[:, None, :]
    return torch.cat([rot, keep], dim=-1)


cos, sin = rope_tables(T)
print(f"cos/sin tables : {list(cos.shape)}  (covering {Rotated_Dimension} dims, not {Head_Dimension})")

_x = torch.randn(T, Heads, Head_Dimension)
_y = apply_rope(_x, cos, sin)

assert torch.equal(_y[..., Rotated_Dimension:], _x[..., Rotated_Dimension:])                                    # tail untouched
assert torch.allclose(_y[0], _x[0], atol=1e-6)                                      # position 0 is identity
assert torch.allclose(_y[..., :Rotated_Dimension].norm(dim=-1), _x[..., :Rotated_Dimension].norm(dim=-1), atol=1e-5)  # rotation is norm-preserving

print(f"  last {Head_Dimension - Rotated_Dimension} dims unchanged elementwise : ok")
print(f"  position 0 (cos=1, sin=0) is the identity : ok")
print(f"  rotated part preserves its norm           : ok")

cos/sin tables : [13, 16]  (covering 16 dims, not 64)
  last 48 dims unchanged elementwise : ok
  position 0 (cos=1, sin=0) is the identity : ok
  rotated part preserves its norm           : ok


In [27]:
layer, head = 0, 0
sl = slice(head * Head_Dimension, (head + 1) * Head_Dimension)

# The block's input: the residual stream through input_layernorm.
x = layer_norm(h_embed, W["attention_ln_weight"][layer], W["attention_ln_bias"][layer])

# 1) Three projections. Weights are [in, out], so this is literally x @ W + b.
q_h = x @ W["W_Q"][layer][:, sl] + W["b_Q"][layer][sl]      # [T, Head_Dimension]
k_h = x @ W["W_K"][layer][:, sl] + W["b_K"][layer][sl]
v_h = x @ W["W_V"][layer][:, sl] + W["b_V"][layer][sl]
print(f"q/k/v : {list(q_h.shape)}")

# 2) RoPE applies to q and k only (never v). apply_rope wants [T, Heads, Head_Dimension], so add a dummy head axis.
q_h = apply_rope(q_h[:, None, :], cos, sin)[:, 0]
k_h = apply_rope(k_h[:, None, :], cos, sin)[:, 0]

# 3) Scaled dot product
scores_h = q_h @ k_h.T / math.sqrt(Head_Dimension)                      # [T, T]

# 4) Causal mask: position i cannot see j > i
causal = torch.full((T, T), float("-inf")).triu(1)
probs_h = (scores_h + causal).softmax(dim=-1)               # [T, T]

assert torch.equal(probs_h.triu(1), torch.zeros(T, T))      # strict upper triangle must be zero
assert torch.allclose(probs_h.sum(-1), torch.ones(T), atol=1e-6)
print(f"scores/probs : {list(probs_h.shape)}   upper triangle zero: ok   rows sum to 1: ok")

# 5) Aggregate the values
z_h = probs_h @ v_h                                         # [T, Head_Dimension]

# 6) Project back. W_O is sliced by ROW, so each head contributes its own [T, Hidden_Dimension] independently.
out_h = z_h @ W["W_O"][layer][sl, :]                        # [T, Hidden_Dimension]
print(f"z : {list(z_h.shape)}  ->  this head's contribution : {list(out_h.shape)}")

print(f"\nlayer {layer} head {head}, attention from the last token:")
for tk, p in zip([tok.decode(i) for i in ids], probs_h[-1].tolist()):
    print(f"  {tk!r:12s} {p:.3f}  {'#' * int(p * 50)}")

q/k/v : [13, 64]
scores/probs : [13, 13]   upper triangle zero: ok   rows sum to 1: ok
z : [13, 64]  ->  this head's contribution : [13, 512]

layer 0 head 0, attention from the last token:
  'The'        0.283  ##############
  ' capital'   0.015  
  ' of'        0.034  #
  ' France'    0.032  #
  ' is'        0.152  #######
  ' Paris'     0.021  #
  ','          0.042  ##
  ' and'       0.060  ##
  ' the'       0.091  ####
  ' capital'   0.027  #
  ' of'        0.076  ###
  ' Germany'   0.013  
  ' is'        0.156  #######


In [28]:
def attention(x, layer, cos, sin, causal):
    """Multi-head causal self-attention for one layer. x is the LayerNormed [T, Hidden_Dimension] input."""
    T = x.shape[0]
    shape = (T, Heads, Head_Dimension)

    q = (x @ W["W_Q"][layer] + W["b_Q"][layer]).view(shape)
    k = (x @ W["W_K"][layer] + W["b_K"][layer]).view(shape)
    v = (x @ W["W_V"][layer] + W["b_V"][layer]).view(shape)

    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    scores = torch.einsum("qnd,knd->nqk", q, k) / math.sqrt(Head_Dimension)     # [Heads, T, T]
    probs = (scores + causal).softmax(dim=-1)

    # Concatenating the heads into [T, Hidden_Dimension] is what makes W_O's input axis the head axis.
    z = torch.einsum("nqk,knd->qnd", probs, v).reshape(T, Hidden_Dimension)
    return z @ W["W_O"][layer] + W["b_O"][layer], probs


attn_out, probs = attention(x, layer, cos, sin, causal)
print(f"attn_out : {list(attn_out.shape)}   probs : {list(probs.shape)}")

# (a) the vectorised head 0 must equal the one unrolled by hand above
assert torch.allclose(probs[head], probs_h, atol=1e-6)
print(f"vectorised head {head} attention == hand-unrolled one   : ok  "
      f"(max diff {(probs[head] - probs_h).abs().max():.1e})")

# (b) concatenate-then-W_O == per-head W_O^h summed, plus b_O
v_all = (x @ W["W_V"][layer] + W["b_V"][layer]).view(T, Heads, Head_Dimension)
by_head = sum((probs[n] @ v_all[:, n]) @ W["W_O"][layer][n * Head_Dimension:(n + 1) * Head_Dimension, :]
              for n in range(Heads)) + W["b_O"][layer]
assert torch.allclose(by_head, attn_out, atol=1e-5)
print(f"concat @ W_O == sum of per-head W_O^h + b_O          : ok  "
      f"(max diff {(by_head - attn_out).abs().max():.1e})")

attn_out : [13, 512]   probs : [8, 13, 13]
vectorised head 0 attention == hand-unrolled one   : ok  (max diff 0.0e+00)
concat @ W_O == sum of per-head W_O^h + b_O          : ok  (max diff 2.4e-06)


## 10. Transformer block (3): the MLP

A two-layer feed-forward net, `512 → 2048 → 512`, with GELU in between.

`hidden_act = "gelu"` means the **exact erf form** $x\,\Phi(x) = \tfrac{x}{2}\left(1 +
\mathrm{erf}(x/\sqrt2)\right)$, not the tanh approximation that many implementations default to. The
cost of getting it wrong is measured below.

In [29]:
def gelu_exact(x):
    """Exact GELU (erf form), not the tanh approximation."""
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))


def mlp(x, layer):
    """One layer's FFN. x is the LayerNormed [T, Hidden_Dimension] input."""
    u = x @ W["W_ffn"][layer] + W["b_ffn"][layer]        # [T, MLP_HH_Dimension] up
    u = gelu_exact(u)
    return u @ W["W_ffn_output"][layer] + W["b_ffn_output"][layer]   # [T, Hidden_Dimension] back down


_t = torch.linspace(-5, 5, 1001)
assert torch.allclose(gelu_exact(_t), F.gelu(_t), atol=1e-6)
print(f"gelu_exact == F.gelu (erf by default) : ok")
print(f"max gap vs the tanh approximation     : {(F.gelu(_t, approximate='tanh') - gelu_exact(_t)).abs().max():.2e}"
      f"   <- no error, just a quiet bias")

mlp_out = mlp(layer_norm(h_embed, W["ffn_ln_weight"][layer], W["ffn_ln_bias"][layer]), layer)
print(f"\nmlp_out : {list(mlp_out.shape)}")

gelu_exact == F.gelu (erf by default) : ok
max gap vs the tanh approximation     : 4.73e-04   <- no error, just a quiet bias

mlp_out : [13, 512]


In [30]:
def block(h, layer, cos, sin, causal):
    """One complete Pythia transformer block (parallel residual)."""
    # The point: both LayerNorms read the same h.
    attn_in = layer_norm(h, W["attention_ln_weight"][layer], W["attention_ln_bias"][layer])
    mlp_in  = layer_norm(h, W["ffn_ln_weight"][layer],       W["ffn_ln_bias"][layer])

    attn_out, probs = attention(attn_in, layer, cos, sin, causal)
    mlp_out = mlp(mlp_in, layer)

    return h + attn_out + mlp_out, probs


def block_serial(h, layer, cos, sin, causal):
    """Counter-example: the GPT-2 style sequential residual. Wrong for Pythia."""
    attn_in = layer_norm(h, W["attention_ln_weight"][layer], W["attention_ln_bias"][layer])
    h1 = h + attention(attn_in, layer, cos, sin, causal)[0]
    mlp_in = layer_norm(h1, W["ffn_ln_weight"][layer], W["ffn_ln_bias"][layer])
    return h1 + mlp(mlp_in, layer)


h_par, _ = block(h_embed, layer, cos, sin, causal)
h_ser    = block_serial(h_embed, layer, cos, sin, causal)

print(f"block output : {list(h_par.shape)}")
print(f"mean residual norm in  : {h_embed.norm(dim=-1).mean():.3f}")
print(f"mean residual norm out : {h_par.norm(dim=-1).mean():.3f}   (a block adds to the stream)")
print(f"\nparallel vs sequential, after a single layer : {(h_par - h_ser).abs().max():.3e} "
      f"({(h_par - h_ser).abs().max() / h_par.abs().max():.1%} relative)")
print("Over 6 layers that compounds to ~5e+01 on the logits and flips the argmax, so the")
print("comparison at the end does catch it.")

block output : [13, 512]
mean residual norm in  : 0.607
mean residual norm out : 7.232   (a block adds to the stream)

parallel vs sequential, after a single layer : 2.552e+00 (47.7% relative)
Over 6 layers that compounds to ~5e+01 on the logits and flips the argmax, so the
comparison at the end does catch it.


In [31]:
def forward(ids, return_cache=False):
    """Run the model over a 1-D tensor of token ids. Returns logits [T, Vocab_Size]."""
    ids = torch.as_tensor(ids, dtype=torch.long).flatten()
    T = ids.shape[0]

    cos, sin = rope_tables(T)
    causal = torch.full((T, T), float("-inf")).triu(1)

    h = W["embed_in"][ids]                       # [T, Hidden_Dimension]
    cache = {"resid": [], "attn_probs": []}

    for layer in range(Layer):
        if return_cache:
            cache["resid"].append(h)             # the INPUT to this layer
        h, probs = block(h, layer, cos, sin, causal)
        if return_cache:
            cache["attn_probs"].append(probs)

    h = layer_norm(h, W["final_ln_weight"], W["final_ln_bias"])
    if return_cache:
        cache["resid"].append(h)                 # last entry is post-final-LN
    logits = h @ W["W_unembed"]                  # [T, Vocab_Size]

    return (logits, cache) if return_cache else logits


logits, cache = forward(ids, return_cache=True)
print(f"logits : {list(logits.shape)}\n")

print(f"prompt: {PROMPT!r}")
print("top-5 next tokens:")
probs_next = logits[-1].softmax(-1)
for score, idx in zip(*logits[-1].topk(5)):
    print(f"  {tok.decode(idx)!r:14s} logit {score:6.2f}   p = {probs_next[idx]:.3f}")

logits : [13, 50304]

prompt: 'The capital of France is Paris, and the capital of Germany is'
top-5 next tokens:
  ' the'         logit 1077.90   p = 0.120
  ' Paris'       logit 1077.61   p = 0.090
  ' Berlin'      logit 1076.35   p = 0.026
  ' Germany'     logit 1075.83   p = 0.015
  ' France'      logit 1075.80   p = 0.015


In [32]:
from transformers import AutoModelForCausalLM

ref = AutoModelForCausalLM.from_pretrained(
    meta["model"], revision=meta["revision"], dtype=torch.float32,
    attn_implementation="eager",     # so output_attentions returns real probability matrices
)
ref.eval()

out = ref(ids[None], output_hidden_states=True, output_attentions=True)


def rel(ours, theirs):
    """Max absolute difference divided by the scale of theirs."""
    return ((ours - theirs).abs().max() / theirs.abs().max().clamp(min=1e-12)).item()


TOL, PROB_TOL_L0 = 1e-4, 1e-5

print("Per-layer (hidden_states[i] is the input to layer i; the last entry is post-final-LN):")
for i, hs in enumerate(out.hidden_states):
    r = rel(cache["resid"][i], hs[0])
    tag = "post final LN" if i == Layer else f"layer {i} input"
    line = f"  {tag:14s}  resid rel {r:.2e}  (scale {hs[0].abs().max():7.2f})"
    if i < Layer:
        d_prob = (cache["attn_probs"][i] - out.attentions[i][0]).abs().max().item()
        line += f"   attn prob abs {d_prob:.2e}"
        if i == 0:
            assert d_prob < PROB_TOL_L0, "layer 0 attention probabilities disagree"
    print(line)
    assert r < TOL, f"residual stream disagrees at entry {i}"

theirs = out.logits[0]
diff = (logits - theirs).abs().max().item()
agree = bool((logits.argmax(-1) == theirs.argmax(-1)).all())

print(f"\nlogits max abs diff : {diff:.2e}   relative {rel(logits, theirs):.2e}   "
      f"(logit scale {theirs.abs().max():.1f})")
print(f"argmax agrees at all {T} positions : {agree}")
print(f"HuggingFace next token : {tok.decode(theirs[-1].argmax())!r}")
print(f"ours next token        : {tok.decode(logits[-1].argmax())!r}")

assert agree and rel(logits, theirs) < TOL
print("\nThe hand-written implementation matches HuggingFace.")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 13263.17it/s]


Per-layer (hidden_states[i] is the input to layer i; the last entry is post-final-LN):
  layer 0 input   resid rel 0.00e+00  (scale    0.31)   attn prob abs 3.87e-06
  layer 1 input   resid rel 1.25e-06  (scale    5.36)   attn prob abs 4.89e-06
  layer 2 input   resid rel 6.19e-07  (scale   12.33)   attn prob abs 1.41e-05
  layer 3 input   resid rel 4.97e-07  (scale   92.18)   attn prob abs 9.77e-04
  layer 4 input   resid rel 5.17e-06  (scale   95.24)   attn prob abs 6.71e-03
  layer 5 input   resid rel 2.04e-05  (scale   42.20)   attn prob abs 5.86e-03
  post final LN   resid rel 8.90e-05  (scale   60.83)

logits max abs diff : 7.32e-03   relative 6.77e-06   (logit scale 1081.5)
argmax agrees at all 13 positions : True
HuggingFace next token : ' the'
ours next token        : ' the'

The hand-written implementation matches HuggingFace.
